In [1]:
# Configuração do ambiente, importações, definição de dispositivo (CPU/CUDA) e caminhos

import sys
import os

# Adiciona o diretório pai (raiz do projeto) ao caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))

from pathlib import Path
import torch

from src.models import MLP  # ajuste conforme o caminho real da sua classe MLP
from src.inference import load_model, predict_regression, predict_classification

MODELS_DIR = Path("../models")

REGRESSION_CHECKPOINT_PATH = MODELS_DIR / "modelo_regressao.pt"
CLASSIFICATION_CHECKPOINT_PATH = MODELS_DIR / "modelo_classificacao.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Carregamento dos arquivos de checkpoint (.pt) dos modelos de regressão e classificação

regression_checkpoint = torch.load(
    REGRESSION_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

classification_checkpoint = torch.load(
    CLASSIFICATION_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

In [3]:
# Inspeção e validação das chaves e metadados contidos em cada checkpoint

print("Checkpoint de regressão:")
print(regression_checkpoint.keys())

print("\nCheckpoint de classificação:")
print(classification_checkpoint.keys())

Checkpoint de regressão:
dict_keys(['task', 'model_state_dict', 'model_config', 'input_dim', 'output_dim', 'preprocessor', 'feature_names', 'target_scaler'])

Checkpoint de classificação:
dict_keys(['task', 'model_state_dict', 'model_config', 'input_dim', 'output_dim', 'preprocessor', 'feature_names', 'classes', 'class_to_index', 'decision_rule'])


In [4]:
# Reconstrução das arquiteturas e carregamento dos pesos nos modelos via load_model

regression_model = load_model(
    regression_checkpoint,
    device,
)

classification_model = load_model(
    classification_checkpoint,
    device,
)

In [5]:
# Exibição da estrutura das camadas das redes neurais reconstruídas a partir dos checkpoints

print("Regressão:")
print(regression_model)

print("\nClassificação:")
print(classification_model)

Regressão:
MLP(
  (network): Sequential(
    (0): Linear(in_features=15, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Classificação:
MLP(
  (network): Sequential(
    (0): Linear(in_features=15, out_features=16, bias=True)
    (1): Tanh()
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): Tanh()
    (4): Linear(in_features=16, out_features=3, bias=True)
  )
)


In [6]:
# Verificação de segurança: confirmação de que os modelos estão em modo de avaliação (eval)

print("Regressão em modo eval:", not regression_model.training)
print("Classificação em modo eval:", not classification_model.training)

Regressão em modo eval: True
Classificação em modo eval: True


In [7]:
# Carregamento das novas amostras tabulares salvas para simulação de inferência em produção

regression_samples = torch.load("../data/processed/regression_samples.pt",
    weights_only=False,
)

classification_samples = torch.load("../data/processed/classification_samples.pt",
    weights_only=False,
)

In [8]:
# Inspeção dos formatos e visualização das amostras de teste carregadas

print("Regressão:")
print(type(regression_samples))
print(regression_samples.shape)
display(regression_samples.head())

print("\nClassificação:")
print(type(classification_samples))
print(classification_samples.shape)
display(classification_samples.head())

Regressão:
<class 'pandas.DataFrame'>
(5, 8)


,Fasting_Blood_Sugar,HbA1c,Age,BMI,Family_History_Diabetes,Hypertension,Diet_Quality,Physical_Activity_Level
33553,192.9,6.7,54.0,37.3,Yes,Yes,Healthy,Low
9427,181.6,5.2,54.0,45.8,No,No,Average,Moderate
199,92.3,6.5,54.0,48.1,No,Yes,Average,Moderate
12447,71.5,4.8,54.0,28.4,Yes,No,Poor,Moderate
39489,127.4,7.9,54.0,28.8,Yes,No,Poor,Moderate



Classificação:
<class 'pandas.DataFrame'>
(5, 9)


,Fasting_Blood_Sugar,HbA1c,Age,BMI,Family_History_Diabetes,Hypertension,Diet_Quality,PCOS,Heart_Disease
6261,183.6,5.4,77.0,19.5,No,No,Average,Yes,No
23351,73.4,8.5,45.0,33.4,No,No,Healthy,No,No
33464,151.0,4.8,40.0,40.3,Yes,Yes,Poor,Yes,No
38347,214.0,12.5,84.0,26.1,Yes,Yes,Healthy,No,Yes
37301,161.6,7.2,22.0,20.4,No,Yes,Poor,No,Yes


In [9]:
# Execução do pipeline de inferência nas amostras (pré-processamento automático e forward pass)

regression_predictions = predict_regression(
    regression_model,
    regression_checkpoint,
    regression_samples,
)

classification_predictions, classification_probabilities = predict_classification(
    classification_model,
    classification_checkpoint,
    classification_samples,
)

In [10]:
# Exibição das previsões finais calculadas para regressão e classificação com probabilidades

print("Previsões de regressão:")
print(regression_predictions)

print("\nPrevisões de classificação:")
print(classification_predictions)

print("\nProbabilidades da classificação:")
print(classification_probabilities)

Previsões de regressão:
[94.782974 61.06295  59.857628 45.88748  85.68744 ]

Previsões de classificação:
['Moderate', 'Moderate', 'High', 'High', 'High']

Probabilidades da classificação:
[[2.5367425e-04 7.7939248e-01 2.2035389e-01]
 [3.9424550e-02 9.3573499e-01 2.4840424e-02]
 [1.7807521e-05 1.2945108e-02 9.8703706e-01]
 [2.5136952e-08 1.2845264e-04 9.9987149e-01]
 [7.4302220e-06 7.9809397e-02 9.2018318e-01]]


In [11]:
# Carregamento dos tensores de saídas originais (em memória antes de salvar) para teste de fidelidade

regression_original_outputs = torch.load(
    "../models/regression_original_outputs.pt",
    map_location="cpu",
    weights_only=True,
)

classification_original_outputs = torch.load(
    "../models/classification_original_outputs.pt",
    map_location="cpu",
    weights_only=True,
)

In [12]:
# Carregamento dos datasets completos de teste processados

PROCESSED_DIR = Path("../data/processed")

regression_datasets = torch.load(
    PROCESSED_DIR / "regression_data.pt",
    weights_only=False,
)

classification_datasets = torch.load(
    PROCESSED_DIR / "classification_data.pt",
    weights_only=False,
)

In [13]:
# Execução de inferência em lote no conjunto de teste completo com os modelos recarregados

X_reg_test = regression_datasets["test"].tensors[0]
X_clf_test = classification_datasets["test"].tensors[0]

regression_model.eval()
classification_model.eval()

with torch.inference_mode():
    regression_loaded_outputs = regression_model(
        X_reg_test.to(device)
    ).cpu()

    classification_loaded_outputs = classification_model(
        X_clf_test.to(device)
    ).cpu()

In [14]:
# Verificação de fidelidade numérica (torch.allclose) e cálculo da diferença absoluta máxima

regression_equal = torch.allclose(
    regression_original_outputs,
    regression_loaded_outputs,
    atol=1e-6,
)

classification_equal = torch.allclose(
    classification_original_outputs,
    classification_loaded_outputs,
    atol=1e-6,
)

regression_max_diff = (
    regression_original_outputs - regression_loaded_outputs
).abs().max().item()

classification_max_diff = (
    classification_original_outputs - classification_loaded_outputs
).abs().max().item()

print("Fidelidade - Regressão:")
print("Resultados equivalentes:", regression_equal)
print("Diferença absoluta máxima:", regression_max_diff)

print("\nFidelidade - Classificação:")
print("Resultados equivalentes:", classification_equal)
print("Diferença absoluta máxima:", classification_max_diff)

Fidelidade - Regressão:
Resultados equivalentes: True
Diferença absoluta máxima: 0.0

Fidelidade - Classificação:
Resultados equivalentes: True
Diferença absoluta máxima: 0.0


In [15]:
# Carregamento dos valores reais de teste para conferência das métricas

y_reg_test = regression_datasets["test"].tensors[1]
y_clf_test = classification_datasets["test"].tensors[1]

print("Regressão:", y_reg_test.shape)
print("Classificação:", y_clf_test.shape)

Regressão: torch.Size([10000, 1])
Classificação: torch.Size([10000])


In [16]:
# Desnormalização do alvo e recálculo das métricas de regressão no teste (RMSE e MAE)

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

regression_target_scaler = regression_checkpoint["target_scaler"]

# Converter previsões para a escala original
regression_predictions = regression_target_scaler.inverse_transform(
    regression_loaded_outputs.numpy()
).ravel()

# Converter valores reais para a escala original
y_reg_original = regression_target_scaler.inverse_transform(
    y_reg_test.numpy()
).ravel()

# Calcular métricas
rmse = np.sqrt(mean_squared_error(y_reg_original, regression_predictions))
mae = mean_absolute_error(y_reg_original, regression_predictions)

print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")

RMSE: 7.661609
MAE: 6.134095


In [17]:
# Recálculo das métricas de classificação no teste (Macro-F1) confirmando reprodutibilidade

from sklearn.metrics import f1_score

# Obter classes previstas
y_clf_pred = classification_loaded_outputs.argmax(dim=1).numpy()

# Valores reais
y_clf_true = y_clf_test.numpy()

# Calcular Macro-F1
macro_f1 = f1_score(
    y_clf_true,
    y_clf_pred,
    average="macro",
)

print(f"Macro-F1: {macro_f1:.6f}")

Macro-F1: 0.702004


In [18]:
# Demonstração de portabilidade: carregamento e execução forçada em CPU (map_location='cpu')
regression_checkpoint_cpu = torch.load(
    REGRESSION_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

classification_checkpoint_cpu = torch.load(
    CLASSIFICATION_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

# Reconstruir os modelos na CPU
regression_model_cpu = load_model(
    regression_checkpoint_cpu,
    device="cpu",
)

classification_model_cpu = load_model(
    classification_checkpoint_cpu,
    device="cpu",
)

print("Dispositivo da regressão:",
      next(regression_model_cpu.parameters()).device)

print("Dispositivo da classificação:",
      next(classification_model_cpu.parameters()).device)

Dispositivo da regressão: cpu
Dispositivo da classificação: cpu


In [19]:
# Análise computacional: contagem de parâmetros treináveis, tamanho teórico e tamanho em disco (KB)
regression_params = sum(
    p.numel() for p in regression_model_cpu.parameters()
    if p.requires_grad
)

classification_params = sum(
    p.numel() for p in classification_model_cpu.parameters()
    if p.requires_grad
)

# Obter tamanho dos arquivos em KB
regression_size_kb = (
    os.path.getsize(REGRESSION_CHECKPOINT_PATH) / 1024
)

classification_size_kb = (
    os.path.getsize(CLASSIFICATION_CHECKPOINT_PATH) / 1024
)

# Estimar tamanho dos parâmetros em float32
regression_params_kb = regression_params * 4 / 1024
classification_params_kb = classification_params * 4 / 1024

print("REGRESSÃO")
print(f"Parâmetros: {regression_params:,}")
print(f"Tamanho teórico dos parâmetros: {regression_params_kb:.2f} KB")
print(f"Tamanho do checkpoint: {regression_size_kb:.2f} KB")

print("\nCLASSIFICAÇÃO")
print(f"Parâmetros: {classification_params:,}")
print(f"Tamanho teórico dos parâmetros: {classification_params_kb:.2f} KB")
print(f"Tamanho do checkpoint: {classification_size_kb:.2f} KB")

REGRESSÃO
Parâmetros: 1,601
Tamanho teórico dos parâmetros: 6.25 KB
Tamanho do checkpoint: 13.81 KB

CLASSIFICAÇÃO
Parâmetros: 579
Tamanho teórico dos parâmetros: 2.26 KB
Tamanho do checkpoint: 9.61 KB
